In [7]:
import os
import json
from typing import TypedDict, List, Dict, Any, Literal
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.checkpoint.memory import MemorySaver
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field

# Load Environment
load_dotenv()

# LLM Gateway Bindings
# We use the fast, cheap model as requested for the router
cheap_model = init_chat_model("google_genai:gemini-2.5-flash")
middle_model = init_chat_model("google_genai:gemini-2.5-flash")
advance_model = init_chat_model("groq:llama-3.3-70b-versatile").with_fallbacks([middle_model, cheap_model])

# Route Model is strictly the lightweight LLM
route_model = cheap_model.with_fallbacks([middle_model, advance_model])

In [8]:
# 1. State Definition
class RouterAgentState(MessagesState):
    user_id: str
    context_summary: str
    selected_agent: str
    routing_reason: str
    execution_logs: List[str]

# 2. Custom Logger Utility
def log_trace(state: RouterAgentState, message: str):
    """Appends a log message to the state and prints it silently in background (we hide prints for clean UI later)."""
    if "execution_logs" not in state or state["execution_logs"] is None:
        state["execution_logs"] = []
    state["execution_logs"].append(message)
    return state

# 3. Mock Semantic Routing Cache
# Bypasses LLM if an exact/similar query was recently routed
ROUTING_CACHE = {}

def check_routing_cache(query: str):
    for cached_query, cached_result in ROUTING_CACHE.items():
        if cached_query.lower() in query.lower() or query.lower() in cached_query.lower():
            return cached_result
    return None

def save_to_routing_cache(query: str, agent: str, reason: str):
    ROUTING_CACHE[query] = {"selected_agent": agent, "routing_reason": f"From Cache: {reason}"}

In [9]:
# 1. Context Retrieval Node
def context_retrieval(state: RouterAgentState):
    log_trace(state, "[Node: context_retrieval] Extracting conversation flow...")
    messages = state["messages"]
    
    # We only care about recent context to decide routing
    history_text = "\n".join([m.content for m in messages[-4:-1]]) if len(messages) > 1 else "No previous history."
    
    state["context_summary"] = history_text
    return {"context_summary": history_text, "execution_logs": state["execution_logs"]}

# 2. Smart Router Node
class RoutingDecision(BaseModel):
    selected_agent: Literal["General Agent", "Lead Agent", "Template Agent", "Campaign Agent", "Research Agent", "Unknown Agent"] = Field(
        description="Select 'General Agent' for greetings, casual chat, or user asset retrieval. Select 'Lead Agent' for searching/gathering target audience or leads. Select 'Template Agent' for drafting cold emails or templates. Select 'Campaign Agent' for launching a campaign or if the user asks for BOTH leads and templates simultaneously. Select 'Research Agent' for external web research. Select 'Unknown Agent' if the request is completely outside the scope of this platform."
    )
    routing_reason: str = Field(description="A brief explanation of why this agent was selected.")

def smart_router_node(state: RouterAgentState):
    user_query = state["messages"][-1].content
    
    # Cache Check
    cached = check_routing_cache(user_query)
    if cached:
        log_trace(state, "[Cache Hit] Routing bypassed LLM.")
        return {"selected_agent": cached["selected_agent"], "routing_reason": cached["routing_reason"], "execution_logs": state["execution_logs"]}
        
    log_trace(state, "[Node: smart_router] Analyzing intent...")
    
    context = state.get("context_summary", "")
    
    prompt = f"""
    You are the Master Dispatcher (Smart Router). Analyze the user query and recent context to determine which sub-agent should handle the request.
    
    RULES:
    1. If the user asks for leads AND templates, default to Campaign Agent (it orchestrates both).
    2. If the user asks to start/launch/send a campaign, select Campaign Agent.
    3. If the user asks to research the web or market, select Research Agent.
    4. If the user asks to find people or leads ONLY, select Lead Agent.
    5. If the user asks to write/draft an email ONLY, select Template Agent.
    6. If the user says hi, or asks general questions about their own assets/profile, select General Agent.\n    7. CRITICAL: If the user asks to perform an action completely unrelated to these agents (e.g. order food, hack a server, write an unrelated app), select 'Unknown Agent' and state that no agent is available for this task.
    
    Context: {context}
    Query: '{user_query}'
    """
    
    structured_llm = route_model.with_structured_output(RoutingDecision)
    
    try:
        decision = structured_llm.invoke([HumanMessage(content=prompt)])
        agent = decision.selected_agent
        reason = decision.routing_reason
        save_to_routing_cache(user_query, agent, reason)
    except Exception as e:
        agent = "General Agent"
        reason = "Fallback applied due to processing error."
        
    log_trace(state, f"[Routing Decision] -> {agent}")
    return {"selected_agent": agent, "routing_reason": reason, "execution_logs": state["execution_logs"]}

# 3. Guardrail Validation Node
def guardrail_validator(state: RouterAgentState):
    """Ensures the LLM didn't hallucinate a fake agent name."""
    valid_agents = ["General Agent", "Lead Agent", "Template Agent", "Campaign Agent", "Research Agent", "Unknown Agent"]
    selected = state.get("selected_agent", "")
    
    if selected not in valid_agents:
        log_trace(state, f"[Guardrail] Invalid agent '{selected}' detected. Defaulting to General Agent.")
        return {"selected_agent": "Unknown Agent", "routing_reason": "Guardrail fallback due to invalid agent name."}
        
    return state

In [10]:
# Compile LangGraph
workflow = StateGraph(RouterAgentState)

workflow.add_node("context", context_retrieval)
workflow.add_node("router", smart_router_node)
workflow.add_node("guardrails", guardrail_validator)

workflow.add_edge(START, "context")
workflow.add_edge("context", "router")
workflow.add_edge("router", "guardrails")
workflow.add_edge("guardrails", END)

memory = MemorySaver()
smart_router_app = workflow.compile(checkpointer=memory)
print("Smart Router Workflow Compiled Successfully!")

Smart Router Workflow Compiled Successfully!


In [11]:
# ==========================================
# Interactive Router Testing Loop
# ==========================================

def run_router_test():
    config = {"configurable": {"thread_id": "router_test_1"}}
    
    print("========================================")
    print("Smart Router - Intent Testing Interface")
    print("Type 'exit', 'quit', or 'bye' to stop.")
    print("NOTE: This will ONLY print the selected agent. It will NOT execute the agent.")
    print("========================================\n")

    while True:
        user_input = input("You: ")
        if user_input.lower() in ['exit', 'quit', 'bye']:
            print("Ending router testing.")
            break
            
        input_dict = {
            "messages": [HumanMessage(content=user_input)],
            "user_id": "test_user_1",
            "execution_logs": []
        }
        
        selected_agent = ""
        reason = ""
        
        for event in smart_router_app.stream(input_dict, config=config):
            for node, values in event.items():
                if node == "guardrails":
                    selected_agent = values["selected_agent"]
                    reason = values["routing_reason"]
                    
        print(f"\n[Selected Agent]: {selected_agent}")
        print(f"[Reason]: {reason}\n")
        print("-----------------------\n")

In [14]:
# run_router_test()